## Translate glosses

1) Read lines from SemCor file
2) Filter by lemmas counts(every lemma was include less than 8 times) 
3) Exclude ambiguous cases and words from validation set
4) Make set of glosses that are used in sentences
5) Translate via Deepseek
6) Save in JSON file

In [1]:
import pandas as pd

df = pd.read_csv('../gold_data/eng_gold.csv')
options = df.word

In [2]:
from nltk import WordNetLemmatizer as wnl

# set of nouns from validation set
test_nouns = set()
for option in options:
    lemma = wnl().lemmatize(option, 'n')
    test_nouns.add(lemma)

In [ ]:
import json

def read_jsonl(file_path, cnt=35_000):
    """
        Read from jsonl file
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        read_lines = 0
        for line in f:
            line = line.strip()
            if line:
                read_lines += 1
                yield json.loads(line)
            if cnt == read_lines:
                break

# Использование:
lines_reader = read_jsonl('../wsd-data/data/jsonl/SemCor.jsonl', cnt=10e6)

In [4]:
lines = []
for line in lines_reader:
    lines.append(line)

In [ ]:
from collections import Counter

cnt = Counter()
train_set = []

for line in lines:
    # filter lines
    if cnt[line['lemma']] < 8 and line['pos'] == 'NOUN' and line['lemma'] not in test_nouns and len(line['sense'].split(';')) == 1:
        train_set.append(line)
        cnt[line['lemma']] += 1

In [3]:
def mask_target(sentence: str, start: int, end: int) -> str:
    """
        Insert placeholders to sentence
    """
    tokens = sentence.split()
    tokens.insert(end, "[TGT]")
    tokens.insert(start, "[TGT]")
    return " ".join(tokens)

from nltk.corpus import wordnet as wn

def get_gloss(sense_key: str) -> str:
    """
        Get gloss from WordNet sense key
    """
    lemma = wn.lemma_from_key(sense_key)
    return lemma.synset().definition()

# gloss_en = "the products of human creativity; works of art collectively"

In [ ]:
def prepare_for_translation(records):
    """
        Apply functuions mask_target and get_gloss for each sentence in filtered lines
    """
    contexts_en, glosses_en = [], []
    for rec in records:
        masked = mask_target(rec["sentence"], rec["start"], rec["end"])
        gloss = get_gloss(rec["sense"])
        contexts_en.append(masked)
        glosses_en.append(gloss)
    return contexts_en, glosses_en

_, glosses = prepare_for_translation(train_set)

In [9]:
glosses_unique = list(set(glosses))

In [6]:
import json

with open("../auxiliary_data/missing_glosses.json", "r") as f:
    missing_glosses = json.load(f)

In [7]:
from openai import OpenAI
from dotenv import load_dotenv
import os
load_dotenv()
key = os.getenv("DEEPSEEK")

client = OpenAI(
    api_key=key,
    base_url="https://api.deepseek.com",
)

In [8]:
import openai

def translate_batch(texts: list[str], source="English", target="Kyrgyz") -> list[str]:
    """
        Make API call to Deepseek to translate batch of glosses
    """
    prompt = (
        f"Translate each of the following English lines to Kyrgyz."
        "Return the translations separated by ' ||| ' in the same order, preserving all punctuation."
    )
    batch_text = " ||| ".join(texts)
    messages = [
        {"role": "system", "content": prompt},
        {"role": "user", "content": batch_text}
    ]

    resp = client.chat.completions.create(
        model="deepseek-chat",
        messages=messages,
        temperature=0.0
    )

    translated = resp.choices[0].message.content.split(" ||| ")

    if len(translated) != len(texts):
        raise ValueError("Количество переводов не совпало с исходным")
    
    translated_batch = [t.strip() for t in translated]
    
    return translated_batch

In [ ]:
from tqdm import trange
from collections import defaultdict

glosses_dct = defaultdict()
problem_index = []
batch_size = 20

for i in trange(0, len(glosses_unique), batch_size):
    try:
        batch = glosses_unique[i:i+batch_size]
        res = translate_batch(batch)
        for eng_gloss, kyr_gloss in zip(batch, res):
            glosses_dct[eng_gloss] = kyr_gloss
    except ValueError as e:
        for eng_gloss in glosses_unique[i:i+batch_size]:
            glosses_dct[eng_gloss] = ""
        problem_index.append(i)


100%|██████████| 607/607 [1:19:15<00:00,  7.84s/it]


In [ ]:
import tqdm

# repeat API calls without batching for problem batches
for problem_ind in tqdm.tqdm(problem_index):
    for i in range(problem_ind, problem_ind + batch_size):
        prompt = (
            "Translate each the following English line to Kyrgyz."
            "Return the translation preserving all punctuation."
        )
        eng_gloss = glosses_unique[i]
        messages = [
            {"role": "system", "content": prompt},
            {"role": "user", "content": eng_gloss}
        ]

        resp = client.chat.completions.create(
            model="deepseek-chat",
            messages=messages,
            temperature=0.0
        )

        kyr_gloss = resp.choices[0].message.content
        glosses_dct[eng_gloss] = kyr_gloss
        

100%|██████████| 66/66 [22:45<00:00, 20.69s/it]


In [10]:
from tqdm import trange
from collections import defaultdict

missing_glosses_dct = defaultdict()
missom_problem_index = []
batch_size = 20

for i in trange(0, len(missing_glosses), batch_size):
    try:
        batch = missing_glosses[i:i+batch_size]
        res = translate_batch(batch)
        for eng_gloss, kyr_gloss in zip(batch, res):
            missing_glosses_dct[eng_gloss] = kyr_gloss
    except ValueError as e:
        for eng_gloss in missing_glosses[i:i+batch_size]:
            missing_glosses_dct[eng_gloss] = ""
        missom_problem_index.append(i)


100%|██████████| 398/398 [1:10:11<00:00, 10.58s/it]


In [11]:
import tqdm

# repeat API calls without batching for problem batches
for problem_ind in tqdm.tqdm(missom_problem_index):
    for i in range(problem_ind, problem_ind + batch_size):
        prompt = (
            "Translate each the following English line to Kyrgyz."
            "Return the translation preserving all punctuation."
        )
        eng_gloss = missing_glosses[i]
        messages = [
            {"role": "system", "content": prompt},
            {"role": "user", "content": eng_gloss}
        ]

        resp = client.chat.completions.create(
            model="deepseek-chat",
            messages=messages,
            temperature=0.0
        )

        kyr_gloss = resp.choices[0].message.content
        missing_glosses_dct[eng_gloss] = kyr_gloss

100%|██████████| 28/28 [12:14<00:00, 26.25s/it]


In [12]:
with open('../auxiliary_data/glosses_dct.json') as f:
    existing = json.load(f)
existing.update(missing_glosses_dct)  # дописываем новые
with open('../auxiliary_data/all_glosses_dct.json', 'w', encoding='utf-8') as f:
    json.dump(existing, f, ensure_ascii=False, indent=4)

In [27]:
with open('glosses_dct.json', 'w', encoding="utf-8") as f:
    json.dump(glosses_dct, f, indent=4, ensure_ascii=False)